In [1]:
#importing the libraries
import pandas as pd
import numpy as np
import pickle
from tensorflow.keras.models import load_model

from src.hybrid import final_hybrid

In [2]:
#loading the datasets
df = pd.read_csv('processed_df.csv')
movies = pd.read_csv('movie.csv')

In [3]:
df.head()

,userId,movieId,rating,timestamp,title,genres,tag,tag_missing,user_avg_rating,movie_avg_rating,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,2,3.5,2005-04-02 23:53:47,Jumanji (1995),Adventure|Children|Fantasy,NaN,1,3.742857,3.213844,...,0,0,0,0,0,0,0,0,0,0
1,1,29,3.5,2005-04-02 23:31:16,"City of Lost Children, The (Cité des enfants p...",Adventure|Drama|Fantasy|Mystery|Sci-Fi,NaN,1,3.742857,3.955775,...,0,0,0,0,1,0,1,0,0,0
2,1,32,3.5,2005-04-02 23:33:39,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller,NaN,1,3.742857,3.902560,...,0,0,0,0,1,0,1,1,0,0
3,1,47,3.5,2005-04-02 23:32:07,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,NaN,1,3.742857,4.056273,...,0,0,0,0,1,0,0,1,0,0
4,1,50,3.5,2005-04-02 23:29:40,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,NaN,1,3.742857,4.334675,...,0,0,0,0,1,0,0,1,0,0


In [4]:
counts = df['tag_missing'].value_counts()

print("Tag Missing (1):", counts.get(1, 0))
print("Tag Present (0):", counts.get(0, 0))

Tag Missing (1): 19874181
Tag Present (0): 391444


In [5]:
df = df.drop(columns=['tag'])

In [6]:
df = df.sort_values('timestamp')

train = df.groupby('userId').head(int(0.8 * len(df) / df['userId'].nunique()))
test = df.drop(train.index)

In [7]:
ground_truth = test.groupby('userId')['movieId'].apply(list).to_dict()

In [8]:
#loading the saved models for evaluation
svd_model = pickle.load(open('models/svd_model.pkl','rb'))
knn_model = pickle.load(open('models/knn_model.pkl','rb'))
xgb_model = pickle.load(open('models/xgb_model.pkl','rb'))

kmeans = pickle.load(open('models/kmeans_model.pkl','rb'))
dbscan = pickle.load(open('models/dbscan_model.pkl','rb'))

lstm_model = load_model('models/lstm_model.h5')
movie_map = pickle.load(open('models/movie_map.pkl','rb'))

feature_cols = pickle.load(open('models/feature_cols.pkl','rb'))
user_features = pd.read_csv('models/user_features.csv', index_col='userId')

In [9]:
train['movie_encoded'] = train['movieId'].map(movie_map)
train['movie_encoded'] = train['movie_encoded'].fillna(0).astype(int)

C:\Users\Pradheesha\AppData\Local\Temp\ipykernel_19980\1573219456.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['movie_encoded'] = train['movieId'].map(movie_map)
C:\Users\Pradheesha\AppData\Local\Temp\ipykernel_19980\1573219456.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['movie_encoded'] = train['movie_encoded'].fillna(0).astype(int)


In [10]:
kmeans_cluster_rating = (
    train.merge(user_features[['cluster']], left_on='userId', right_index=True)
         .groupby(['cluster', 'movieId'])['rating']
         .mean()
         .to_dict()
)

dbscan_cluster_rating = (
    train.merge(user_features[['db_cluster']], left_on='userId', right_index=True)
         .groupby(['db_cluster', 'movieId'])['rating']
         .mean()
         .to_dict()
)

In [11]:
def precision_recall_at_k(recommended, relevant):

    if len(recommended) == 0 or len(relevant) == 0:
        return 0, 0

    recommended = set(recommended)
    relevant = set(relevant)

    tp = len(recommended & relevant)

    precision = tp / len(recommended)
    recall = tp / len(relevant)

    return precision, recall

In [13]:
from tqdm import tqdm
precisions, recalls = [], []

users = list(ground_truth.keys())[:100]  # limit for speed

for user in tqdm(users):
    
    recs = final_hybrid(user_id=user, df=train, movies=movies,
                    svd_model=svd_model, knn_model=knn_model, xgb_model=xgb_model,
                    lstm_model=lstm_model, movie_map=movie_map,
                    user_features=user_features, feature_cols=feature_cols,
                    kmeans_cluster_rating=kmeans_cluster_rating,
                    dbscan_cluster_rating=dbscan_cluster_rating,
                    n=10)
    
    if recs.empty:
        continue

    recommended = recs['movieId'].tolist()
    relevant = ground_truth.get(user, [])

    p, r = precision_recall_at_k(recommended, relevant)

    precisions.append(p)
    recalls.append(r)

100%|██████████| 100/100 [1:45:47<00:00, 63.48s/it]


In [14]:
print("Precision@10:", np.mean(precisions))
print("Recall@10:", np.mean(recalls))

Precision@10: 0.17300000000000004
Recall@10: 0.01355711070140591
